In [12]:
# 1 - Load data
# 2 - Prepare it
# 3 - Construct domains
# 4 - Process domains as needed
# 5 - Define metrics and estimation methods
# 6 - Visualize domains

In [13]:
"""
Define configuration.
Load both CSVs.
identify metadata and feature columns.
Define experiment-specific label mappings.
Construct source and target domains.
Validate domain compatibility.
Generate sample and class summaries.
Fit source-based standardization.
Produce PCA visualizations.
Analyze class proportions.
Calculate marginal distances.
Calculate class-conditional distances.
Run domain-classification analysis.
Store plot-ready and tabular results.
"""

'\nDefine configuration.\nLoad both CSVs.\nidentify metadata and feature columns.\nDefine experiment-specific label mappings.\nConstruct source and target domains.\nValidate domain compatibility.\nGenerate sample and class summaries.\nFit source-based standardization.\nProduce PCA visualizations.\nAnalyze class proportions.\nCalculate marginal distances.\nCalculate class-conditional distances.\nRun domain-classification analysis.\nStore plot-ready and tabular results.\n'

In [14]:
import sys
import os
from pathlib import Path

import pandas as pd

sys.path.append(os.path.abspath("../../../"))
PROJECT_ROOT = Path("../../../").resolve()

import preprocessing.general.feature_extraction as fe


# ============================================================
# General configuration
# ============================================================

ELECTRODE_SETUP = "setup_01"

METADATA_COLUMNS = [
    "dataset",
    "subject",
    "session",
    "trial_index",
    "label",
]

TRIAL_IDENTIFIER_COLUMNS = [
    "dataset",
    "subject",
    "session",
    "trial_index",
]


# ============================================================
# Input paths
# ============================================================

bci_csv = (
    PROJECT_ROOT
    / "Datasets/MotorImagery/processed"
    / f"bci_features_{ELECTRODE_SETUP}.csv"
)

physionet_csv = (
    PROJECT_ROOT
    / "Datasets/EEG Motor Movement/processed"
    / f"EEG_MM_features_{ELECTRODE_SETUP}.csv"
)


# ============================================================
# Check files
# ============================================================

if not bci_csv.exists():
    raise FileNotFoundError(
        f"BCI feature file not found: {bci_csv}"
    )

if not physionet_csv.exists():
    raise FileNotFoundError(
        f"PhysioNet feature file not found: {physionet_csv}"
    )


# ============================================================
# Load datasets
# ============================================================

bci_df = pd.read_csv(bci_csv)
physionet_df = pd.read_csv(physionet_csv)

print(f"BCI shape: {bci_df.shape}")
print(f"PhysioNet shape: {physionet_df.shape}")


# ============================================================
# Validate individual datasets
# ============================================================

fe.validate_feature_dataframe(bci_df)
fe.validate_feature_dataframe(physionet_df)

if set(bci_df["dataset"].unique()) != {"bci_iv_2a"}:
    raise ValueError(
        "Unexpected dataset identifier in the BCI CSV: "
        f"{bci_df['dataset'].unique().tolist()}"
    )

if set(physionet_df["dataset"].unique()) != {"eegmmidb"}:
    raise ValueError(
        "Unexpected dataset identifier in the PhysioNet CSV: "
        f"{physionet_df['dataset'].unique().tolist()}"
    )

print("✅ Individual dataset validation passed.")


# ============================================================
# Identify feature columns
# ============================================================

bci_feature_columns = [
    column
    for column in bci_df.columns
    if column not in METADATA_COLUMNS
]

physionet_feature_columns = [
    column
    for column in physionet_df.columns
    if column not in METADATA_COLUMNS
]

common_feature_columns = [
    column
    for column in bci_feature_columns
    if column in physionet_feature_columns
]

bci_only_feature_columns = [
    column
    for column in bci_feature_columns
    if column not in physionet_feature_columns
]

physionet_only_feature_columns = [
    column
    for column in physionet_feature_columns
    if column not in bci_feature_columns
]


# ============================================================
# Report feature compatibility
# ============================================================

print("\nFeature schema")
print(f"BCI features: {len(bci_feature_columns)}")
print(
    f"PhysioNet features: "
    f"{len(physionet_feature_columns)}"
)
print(f"Common features: {len(common_feature_columns)}")
print(
    f"BCI-only features: "
    f"{len(bci_only_feature_columns)}"
)
print(
    f"PhysioNet-only features: "
    f"{len(physionet_only_feature_columns)}"
)

if bci_only_feature_columns:
    print(
        "BCI-only examples:",
        bci_only_feature_columns[:10],
    )

if physionet_only_feature_columns:
    print(
        "PhysioNet-only examples:",
        physionet_only_feature_columns[:10],
    )


# ============================================================
# Create union of feature columns
# ============================================================

feature_columns = bci_feature_columns.copy()

feature_columns.extend([
    column
    for column in physionet_feature_columns
    if column not in feature_columns
])


# ============================================================
# Merge datasets
# ============================================================

combined_df = pd.concat(
    [
        bci_df,
        physionet_df,
    ],
    ignore_index=True,
    sort=False,
)

# Missing features from one dataset remain NaN.
combined_df = combined_df[
    METADATA_COLUMNS + feature_columns
]


# ============================================================
# Validate merged metadata
# ============================================================

if combined_df[METADATA_COLUMNS].isna().any().any():
    invalid_columns = (
        combined_df[METADATA_COLUMNS]
        .columns[
            combined_df[METADATA_COLUMNS]
            .isna()
            .any()
        ]
        .tolist()
    )

    raise ValueError(
        "Missing metadata after merging in columns: "
        f"{invalid_columns}"
    )

duplicate_trials = combined_df.duplicated(
    subset=TRIAL_IDENTIFIER_COLUMNS,
    keep=False,
)

if duplicate_trials.any():
    examples = (
        combined_df.loc[
            duplicate_trials,
            TRIAL_IDENTIFIER_COLUMNS,
        ]
        .drop_duplicates()
        .head(10)
        .to_dict("records")
    )

    raise ValueError(
        "Duplicated trial identifiers after merging. "
        f"Examples: {examples}"
    )


# ============================================================
# Report merged data
# ============================================================

print(
    f"\n✅ Combined dataset created: "
    f"{combined_df.shape}"
)

print(
    f"✅ Datasets: "
    f"{combined_df['dataset'].unique().tolist()}"
)

print(
    f"✅ Total subjects: "
    f"{combined_df.groupby('dataset')['subject'].nunique().sum()}"
)

display(combined_df.head())

BCI shape: (5184, 95)
PhysioNet shape: (19673, 95)
✅ Individual dataset validation passed.

Feature schema
BCI features: 90
PhysioNet features: 90
Common features: 90
BCI-only features: 0
PhysioNet-only features: 0

✅ Combined dataset created: (24857, 95)
✅ Datasets: ['bci_iv_2a', 'eegmmidb']
✅ Total subjects: 118


,dataset,subject,session,trial_index,label,b8_12_mean_C3,b8_12_mean_Cz,b8_12_mean_C4,b8_12_std_C3,b8_12_std_Cz,...,b13_30_q_1_C4,b13_30_q_2_C3,b13_30_q_2_Cz,b13_30_q_2_C4,b13_30_q_3_C3,b13_30_q_3_Cz,b13_30_q_3_C4,b13_30_logvar_C3,b13_30_logvar_Cz,b13_30_logvar_C4
0,bci_iv_2a,A01,session_01,0,tongue_imagery,6.628982e-09,-2.770257e-08,8.171160e-10,0.000002,0.000003,...,2.083999e-08,3.014338e-08,3.604521e-08,-2.125241e-08,-5.205597e-09,-2.993853e-08,-3.390606e-08,-25.160662,-24.961687,-25.039858
1,bci_iv_2a,A01,session_01,1,both_feet_imagery,5.556458e-08,5.848129e-08,5.225703e-08,0.000004,0.000004,...,8.526831e-08,-1.014640e-08,-3.665703e-08,-2.253099e-08,8.930448e-08,1.509851e-07,1.000519e-07,-25.358910,-25.109524,-24.992792
2,bci_iv_2a,A01,session_01,2,right_hand_imagery,4.961544e-09,1.173579e-08,2.111302e-08,0.000002,0.000003,...,6.735552e-08,-5.209813e-09,-5.477280e-09,2.947687e-08,-1.845828e-08,-1.485296e-08,-1.963981e-08,-25.272942,-25.033789,-25.192879
3,bci_iv_2a,A01,session_01,3,left_hand_imagery,-8.338073e-08,-6.672642e-08,-3.774150e-08,0.000003,0.000003,...,5.061478e-08,-9.388884e-09,-5.350012e-08,-4.387143e-08,-2.301171e-08,1.181291e-08,6.863667e-11,-25.134298,-25.061962,-25.328821
4,bci_iv_2a,A01,session_01,4,left_hand_imagery,3.510155e-09,-8.964781e-09,-6.915561e-09,0.000002,0.000002,...,-2.910713e-08,-5.453906e-08,-9.668239e-08,-2.637220e-08,7.831463e-08,1.139543e-07,1.991119e-08,-25.495483,-25.204264,-25.121435


In [15]:
# ============================================================
# Initial data filters
# ============================================================

# None means that no filter is applied.
# Values may be scalars or lists.
DATA_FILTERS = {
    "dataset": None,
    "subject": None,
    "session": None,
    "label": None,
}

# Examples:
#
# Only BCI:
# DATA_FILTERS["dataset"] = "bci_iv_2a"
#
# Only selected labels:
# DATA_FILTERS["label"] = [
#     "left_hand_imagery",
#     "right_hand_imagery",
# ]


# ============================================================
# Elementary-domain definition
# ============================================================

# These columns jointly define one elementary domain.
DOMAIN_COLUMNS = [
    "dataset",
    "subject",
    "session",
]

# Other possibilities:
#
# One domain per dataset:
# DOMAIN_COLUMNS = ["dataset"]
#
# One domain per subject:
# DOMAIN_COLUMNS = ["subject"]
#
# One domain per subject-session pair:
# DOMAIN_COLUMNS = ["subject", "session"]
#
# One domain per dataset-subject-session:
# DOMAIN_COLUMNS = [
#     "dataset",
#     "subject",
#     "session",
# ]


# ============================================================
# Filtering function
# ============================================================

def apply_dataframe_filters(
    dataframe,
    filters,
):
    """
    Filter a DataFrame using scalar values, collections, or None.
    """
    filtered = dataframe.copy()

    for column, values in filters.items():
        if values is None:
            continue

        if column not in filtered.columns:
            raise KeyError(
                f"Filter column not found: {column}"
            )

        if isinstance(values, str) or not isinstance(
            values,
            (list, tuple, set),
        ):
            values = [values]

        filtered = filtered.loc[
            filtered[column].isin(values)
        ]

    return filtered.reset_index(drop=True)


# ============================================================
# Apply filters
# ============================================================

domain_df = apply_dataframe_filters(
    dataframe=combined_df,
    filters=DATA_FILTERS,
)

if domain_df.empty:
    raise ValueError(
        "No rows remain after applying DATA_FILTERS."
    )


# ============================================================
# Validate domain definition
# ============================================================

if not DOMAIN_COLUMNS:
    raise ValueError(
        "DOMAIN_COLUMNS cannot be empty."
    )

missing_domain_columns = [
    column
    for column in DOMAIN_COLUMNS
    if column not in domain_df.columns
]

if missing_domain_columns:
    raise ValueError(
        "Missing domain-definition columns: "
        f"{missing_domain_columns}"
    )

if domain_df[DOMAIN_COLUMNS].isna().any().any():
    raise ValueError(
        "Missing values were found in DOMAIN_COLUMNS."
    )


# ============================================================
# Create elementary-domain identifiers
# ============================================================

domain_df["domain_id"] = (
    domain_df[DOMAIN_COLUMNS]
    .astype(str)
    .agg("_".join, axis=1)
)


# ============================================================
# Reorder columns
# ============================================================

domain_metadata_columns = [
    "dataset",
    "domain_id",
    "subject",
    "session",
    "trial_index",
    "label",
]

domain_df = domain_df[
    domain_metadata_columns + feature_columns
]


# ============================================================
# Create domain summary
# ============================================================

domain_summary = (
    domain_df
    .groupby(
        "domain_id",
        as_index=False,
    )
    .agg(
        n_trials=("label", "size"),
        n_datasets=("dataset", "nunique"),
        n_subjects=("subject", "nunique"),
        n_sessions=("session", "nunique"),
        n_labels=("label", "nunique"),
    )
    .sort_values("domain_id")
    .reset_index(drop=True)
)


# ============================================================
# Report
# ============================================================

print(
    f"✅ Rows after filtering: {len(domain_df)}"
)

print(
    f"✅ Elementary domains: "
    f"{domain_df['domain_id'].nunique()}"
)

print(
    f"✅ Domain definition: {DOMAIN_COLUMNS}"
)

display(domain_summary.head(20))

✅ Rows after filtering: 24857
✅ Elementary domains: 127
✅ Domain definition: ['dataset', 'subject', 'session']


,domain_id,n_trials,n_datasets,n_subjects,n_sessions,n_labels
0,bci_iv_2a_A01_session_01,288,1,1,1,4
1,bci_iv_2a_A01_session_02,288,1,1,1,4
2,bci_iv_2a_A02_session_01,288,1,1,1,4
3,bci_iv_2a_A02_session_02,288,1,1,1,4
4,bci_iv_2a_A03_session_01,288,1,1,1,4
5,bci_iv_2a_A03_session_02,288,1,1,1,4
6,bci_iv_2a_A04_session_01,288,1,1,1,4
7,bci_iv_2a_A04_session_02,288,1,1,1,4
8,bci_iv_2a_A05_session_01,288,1,1,1,4
9,bci_iv_2a_A05_session_02,288,1,1,1,4


within_subject_pooled

within_session

cross_dataset_leave_one_out

cross_dataset_pairwise

cross_session_pooled

cross_session_within_subject

leave_one_subject_out

pairwise_subjects

increasing_source_domains

In [16]:
# ============================================================
# Common split helpers
# ============================================================

from itertools import permutations

import numpy as np


AVAILABLE_DATASETS = sorted(
    domain_df["dataset"]
    .unique()
    .tolist()
)


def select_datasets(
    dataframe,
    datasets=None,
):
    """
    Select the datasets used by a domain-split strategy.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Domain DataFrame.

    datasets : str, list of str, or None
        Datasets to retain. None retains all available datasets.
    """
    if datasets is None:
        selected = dataframe.copy()

    else:
        if isinstance(datasets, str):
            datasets = [datasets]

        unknown_datasets = (
            set(datasets)
            - set(dataframe["dataset"].unique())
        )

        if unknown_datasets:
            raise ValueError(
                "Unknown datasets: "
                f"{sorted(unknown_datasets)}"
            )

        selected = dataframe.loc[
            dataframe["dataset"].isin(datasets)
        ].copy()

    if selected.empty:
        raise ValueError(
            "No rows remain after selecting datasets."
        )

    return selected.reset_index(drop=True)


def get_domain_ids(
    dataframe,
    dataset=None,
    subject=None,
    session=None,
):
    """
    Return domain IDs matching the supplied metadata values.
    """
    filters = {
        "dataset": dataset,
        "subject": subject,
        "session": session,
    }

    selected = apply_dataframe_filters(
        dataframe=dataframe,
        filters=filters,
    )

    return sorted(
        selected["domain_id"]
        .unique()
        .tolist()
    )


def create_split_record(
    strategy,
    split_id,
    source_domain_ids=None,
    target_domain_ids=None,
    shared_domain_ids=None,
    metadata=None,
    seed=None,
    repetition=None,
):
    """
    Create one standardized domain-split record.

    For ordinary domain-transfer strategies, source_domain_ids
    and target_domain_ids contain complete elementary domains.

    For within-domain strategies, shared_domain_ids contains the
    domains whose trials will later be divided into source and
    target subsets.
    """
    source_domain_ids = sorted(
        set(source_domain_ids or [])
    )

    target_domain_ids = sorted(
        set(target_domain_ids or [])
    )

    shared_domain_ids = sorted(
        set(shared_domain_ids or [])
    )

    is_within_domain = bool(shared_domain_ids)

    if is_within_domain:
        if source_domain_ids or target_domain_ids:
            raise ValueError(
                "Within-domain records should use only "
                "shared_domain_ids."
            )

    else:
        if not source_domain_ids:
            raise ValueError(
                f"Split '{split_id}' has no source domains."
            )

        if not target_domain_ids:
            raise ValueError(
                f"Split '{split_id}' has no target domains."
            )

        overlapping_domains = (
            set(source_domain_ids)
            & set(target_domain_ids)
        )

        if overlapping_domains:
            raise ValueError(
                f"Split '{split_id}' contains overlapping "
                f"source and target domains: "
                f"{sorted(overlapping_domains)}"
            )

    return {
        "split_id": split_id,
        "strategy": strategy,
        "source_domain_ids": source_domain_ids,
        "target_domain_ids": target_domain_ids,
        "shared_domain_ids": shared_domain_ids,
        "requires_trial_split": is_within_domain,
        "n_source_domains": len(source_domain_ids),
        "n_target_domains": len(target_domain_ids),
        "n_shared_domains": len(shared_domain_ids),
        "seed": seed,
        "repetition": repetition,
        "metadata": metadata or {},
    }


def validate_domain_splits(
    splits,
    dataframe,
):
    """
    Validate a list of generated split records.
    """
    if not splits:
        raise ValueError(
            "No domain splits were generated."
        )

    available_domain_ids = set(
        dataframe["domain_id"].unique()
    )

    split_ids = [
        split["split_id"]
        for split in splits
    ]

    duplicated_split_ids = sorted({
        split_id
        for split_id in split_ids
        if split_ids.count(split_id) > 1
    })

    if duplicated_split_ids:
        raise ValueError(
            "Duplicated split identifiers found: "
            f"{duplicated_split_ids[:10]}"
        )

    for split in splits:
        referenced_domains = (
            set(split["source_domain_ids"])
            | set(split["target_domain_ids"])
            | set(split["shared_domain_ids"])
        )

        unknown_domains = (
            referenced_domains
            - available_domain_ids
        )

        if unknown_domains:
            raise ValueError(
                f"Split '{split['split_id']}' references "
                f"unknown domains: {sorted(unknown_domains)}"
            )

    return True


def create_split_summary(
    splits,
):
    """
    Convert generated split records into a summary DataFrame.
    """
    rows = []

    for split in splits:
        row = {
            "split_id": split["split_id"],
            "strategy": split["strategy"],
            "n_source_domains": split["n_source_domains"],
            "n_target_domains": split["n_target_domains"],
            "n_shared_domains": split["n_shared_domains"],
            "requires_trial_split": (
                split["requires_trial_split"]
            ),
            "seed": split["seed"],
            "repetition": split["repetition"],
        }

        row.update(split["metadata"])
        rows.append(row)

    return pd.DataFrame(rows)


def materialize_domain_split(
    dataframe,
    split,
):
    """
    Materialize one split.

    Ordinary strategies return source_df and target_df.

    Within-domain strategies return within_df because their
    source-target trial division belongs to the later training
    and evaluation stage.
    """
    if split["requires_trial_split"]:
        within_df = (
            dataframe.loc[
                dataframe["domain_id"].isin(
                    split["shared_domain_ids"]
                )
            ]
            .copy()
            .reset_index(drop=True)
        )

        return {
            "source_df": None,
            "target_df": None,
            "within_df": within_df,
        }

    source_df = (
        dataframe.loc[
            dataframe["domain_id"].isin(
                split["source_domain_ids"]
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    target_df = (
        dataframe.loc[
            dataframe["domain_id"].isin(
                split["target_domain_ids"]
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    return {
        "source_df": source_df,
        "target_df": target_df,
        "within_df": None,
    }


print(
    f"✅ Available datasets: {AVAILABLE_DATASETS}"
)

✅ Available datasets: ['bci_iv_2a', 'eegmmidb']


In [17]:
# ============================================================
# Within-domain strategies
# ============================================================

def generate_within_subject_pooled_splits(
    dataframe,
    datasets=None,
):
    """
    Create one split specification per subject.

    All sessions belonging to the subject are pooled. The trials
    will later be divided into source and target subsets.
    """
    selected_df = select_datasets(
        dataframe=dataframe,
        datasets=datasets,
    )

    subject_units = (
        selected_df[
            [
                "dataset",
                "subject",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "dataset",
                "subject",
            ]
        )
    )

    splits = []

    for dataset, subject in subject_units.itertuples(
        index=False,
        name=None,
    ):
        shared_domain_ids = get_domain_ids(
            dataframe=selected_df,
            dataset=dataset,
            subject=subject,
        )

        split_id = (
            f"within_subject_pooled__"
            f"{dataset}__{subject}"
        )

        splits.append(
            create_split_record(
                strategy="within_subject_pooled",
                split_id=split_id,
                shared_domain_ids=shared_domain_ids,
                metadata={
                    "dataset": dataset,
                    "subject": subject,
                    "n_sessions": len(
                        selected_df.loc[
                            (
                                selected_df["dataset"]
                                == dataset
                            )
                            & (
                                selected_df["subject"]
                                == subject
                            ),
                            "session",
                        ].unique()
                    ),
                },
            )
        )

    return splits


def generate_within_session_splits(
    dataframe,
    datasets=None,
):
    """
    Create one split specification per subject-session domain.

    Trials from the same session will later be divided into
    source and target subsets.
    """
    selected_df = select_datasets(
        dataframe=dataframe,
        datasets=datasets,
    )

    session_units = (
        selected_df[
            [
                "dataset",
                "subject",
                "session",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "dataset",
                "subject",
                "session",
            ]
        )
    )

    splits = []

    for (
        dataset,
        subject,
        session,
    ) in session_units.itertuples(
        index=False,
        name=None,
    ):
        shared_domain_ids = get_domain_ids(
            dataframe=selected_df,
            dataset=dataset,
            subject=subject,
            session=session,
        )

        split_id = (
            f"within_session__"
            f"{dataset}__{subject}__{session}"
        )

        splits.append(
            create_split_record(
                strategy="within_session",
                split_id=split_id,
                shared_domain_ids=shared_domain_ids,
                metadata={
                    "dataset": dataset,
                    "subject": subject,
                    "session": session,
                },
            )
        )

    return splits

In [18]:
# ============================================================
# Dataset-level strategies
# ============================================================

def generate_cross_dataset_leave_one_out_splits(
    dataframe,
    datasets=None,
):
    """
    Use one complete dataset as target and every other selected
    dataset as source.

    With datasets A, B, and C:

        A + B -> C
        A + C -> B
        B + C -> A
    """
    selected_df = select_datasets(
        dataframe=dataframe,
        datasets=datasets,
    )

    dataset_values = sorted(
        selected_df["dataset"]
        .unique()
        .tolist()
    )

    if len(dataset_values) < 2:
        raise ValueError(
            "Cross-dataset analysis requires at least "
            "two datasets."
        )

    splits = []

    for target_dataset in dataset_values:
        source_datasets = [
            dataset
            for dataset in dataset_values
            if dataset != target_dataset
        ]

        source_domain_ids = get_domain_ids(
            dataframe=selected_df.loc[
                selected_df["dataset"].isin(
                    source_datasets
                )
            ]
        )

        target_domain_ids = get_domain_ids(
            dataframe=selected_df,
            dataset=target_dataset,
        )

        split_id = (
            "cross_dataset_leave_one_out__"
            f"target_{target_dataset}"
        )

        splits.append(
            create_split_record(
                strategy=(
                    "cross_dataset_leave_one_out"
                ),
                split_id=split_id,
                source_domain_ids=source_domain_ids,
                target_domain_ids=target_domain_ids,
                metadata={
                    "source_datasets": ",".join(
                        source_datasets
                    ),
                    "target_dataset": target_dataset,
                },
            )
        )

    return splits


def generate_cross_dataset_pairwise_splits(
    dataframe,
    datasets=None,
):
    """
    Generate every ordered dataset transfer pair.

    With datasets A, B, and C:

        A -> B
        B -> A
        A -> C
        C -> A
        B -> C
        C -> B
    """
    selected_df = select_datasets(
        dataframe=dataframe,
        datasets=datasets,
    )

    dataset_values = sorted(
        selected_df["dataset"]
        .unique()
        .tolist()
    )

    if len(dataset_values) < 2:
        raise ValueError(
            "Cross-dataset pairwise analysis requires "
            "at least two datasets."
        )

    splits = []

    for (
        source_dataset,
        target_dataset,
    ) in permutations(
        dataset_values,
        2,
    ):
        source_domain_ids = get_domain_ids(
            dataframe=selected_df,
            dataset=source_dataset,
        )

        target_domain_ids = get_domain_ids(
            dataframe=selected_df,
            dataset=target_dataset,
        )

        split_id = (
            "cross_dataset_pairwise__"
            f"{source_dataset}_to_{target_dataset}"
        )

        splits.append(
            create_split_record(
                strategy="cross_dataset_pairwise",
                split_id=split_id,
                source_domain_ids=source_domain_ids,
                target_domain_ids=target_domain_ids,
                metadata={
                    "source_dataset": source_dataset,
                    "target_dataset": target_dataset,
                },
            )
        )

    return splits

In [19]:
# ============================================================
# Subject-level strategies
# ============================================================

def _get_subject_units(
    dataframe,
):
    """
    Return unique dataset-subject units.
    """
    return (
        dataframe[
            [
                "dataset",
                "subject",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "dataset",
                "subject",
            ]
        )
        .reset_index(drop=True)
    )


def generate_leave_one_subject_out_splits(
    dataframe,
    datasets=None,
    same_dataset_only=False,
):
    """
    Use one subject as target and every other eligible subject
    as source.

    When same_dataset_only=True, sources must come from the same
    dataset as the target subject.
    """
    selected_df = select_datasets(
        dataframe=dataframe,
        datasets=datasets,
    )

    subject_units = _get_subject_units(selected_df)
    splits = []

    for (
        target_dataset,
        target_subject,
    ) in subject_units.itertuples(
        index=False,
        name=None,
    ):
        candidate_sources = subject_units.loc[
            ~(
                (
                    subject_units["dataset"]
                    == target_dataset
                )
                & (
                    subject_units["subject"]
                    == target_subject
                )
            )
        ].copy()

        if same_dataset_only:
            candidate_sources = (
                candidate_sources.loc[
                    candidate_sources["dataset"]
                    == target_dataset
                ]
            )

        if candidate_sources.empty:
            continue

        source_domain_ids = []

        for (
            source_dataset,
            source_subject,
        ) in candidate_sources.itertuples(
            index=False,
            name=None,
        ):
            source_domain_ids.extend(
                get_domain_ids(
                    dataframe=selected_df,
                    dataset=source_dataset,
                    subject=source_subject,
                )
            )

        target_domain_ids = get_domain_ids(
            dataframe=selected_df,
            dataset=target_dataset,
            subject=target_subject,
        )

        split_id = (
            "leave_one_subject_out__"
            f"target_{target_dataset}_{target_subject}"
        )

        splits.append(
            create_split_record(
                strategy="leave_one_subject_out",
                split_id=split_id,
                source_domain_ids=source_domain_ids,
                target_domain_ids=target_domain_ids,
                metadata={
                    "target_dataset": target_dataset,
                    "target_subject": target_subject,
                    "same_dataset_only": (
                        same_dataset_only
                    ),
                    "n_source_subjects": len(
                        candidate_sources
                    ),
                },
            )
        )

    return splits


def generate_pairwise_subject_splits(
    dataframe,
    datasets=None,
    same_dataset_only=False,
    max_pairs=None,
    seed=42,
):
    """
    Generate ordered source-subject -> target-subject pairs.

    max_pairs may be used to randomly restrict the number of
    generated pairs.
    """
    selected_df = select_datasets(
        dataframe=dataframe,
        datasets=datasets,
    )

    subject_units = list(
        _get_subject_units(selected_df)
        .itertuples(
            index=False,
            name=None,
        )
    )

    subject_pairs = []

    for source_unit in subject_units:
        for target_unit in subject_units:
            if source_unit == target_unit:
                continue

            if (
                same_dataset_only
                and source_unit[0] != target_unit[0]
            ):
                continue

            subject_pairs.append(
                (
                    source_unit,
                    target_unit,
                )
            )

    if max_pairs is not None:
        if max_pairs <= 0:
            raise ValueError(
                "max_pairs must be positive."
            )

        if len(subject_pairs) > max_pairs:
            rng = np.random.default_rng(seed)

            selected_indices = rng.choice(
                len(subject_pairs),
                size=max_pairs,
                replace=False,
            )

            subject_pairs = [
                subject_pairs[index]
                for index in selected_indices
            ]

    splits = []

    for (
        (
            source_dataset,
            source_subject,
        ),
        (
            target_dataset,
            target_subject,
        ),
    ) in subject_pairs:
        source_domain_ids = get_domain_ids(
            dataframe=selected_df,
            dataset=source_dataset,
            subject=source_subject,
        )

        target_domain_ids = get_domain_ids(
            dataframe=selected_df,
            dataset=target_dataset,
            subject=target_subject,
        )

        split_id = (
            "pairwise_subjects__"
            f"{source_dataset}_{source_subject}"
            f"_to_"
            f"{target_dataset}_{target_subject}"
        )

        splits.append(
            create_split_record(
                strategy="pairwise_subjects",
                split_id=split_id,
                source_domain_ids=source_domain_ids,
                target_domain_ids=target_domain_ids,
                seed=seed,
                metadata={
                    "source_dataset": source_dataset,
                    "source_subject": source_subject,
                    "target_dataset": target_dataset,
                    "target_subject": target_subject,
                    "same_dataset_only": (
                        same_dataset_only
                    ),
                },
            )
        )

    return splits


def generate_increasing_source_domain_splits(
    dataframe,
    datasets=None,
    same_dataset_only=False,
    n_repetitions=1,
    min_sources=1,
    max_sources=None,
    seed=42,
):
    """
    Fix one target subject and progressively add source subjects.

    Each repetition uses a different randomized but nested source
    ordering:

        {S1} -> T
        {S1, S2} -> T
        {S1, S2, S3} -> T
        ...
    """
    selected_df = select_datasets(
        dataframe=dataframe,
        datasets=datasets,
    )

    subject_units = _get_subject_units(selected_df)
    splits = []

    for target_index, (
        target_dataset,
        target_subject,
    ) in enumerate(
        subject_units.itertuples(
            index=False,
            name=None,
        )
    ):
        candidate_sources = subject_units.loc[
            ~(
                (
                    subject_units["dataset"]
                    == target_dataset
                )
                & (
                    subject_units["subject"]
                    == target_subject
                )
            )
        ].copy()

        if same_dataset_only:
            candidate_sources = (
                candidate_sources.loc[
                    candidate_sources["dataset"]
                    == target_dataset
                ]
            )

        candidate_sources = list(
            candidate_sources.itertuples(
                index=False,
                name=None,
            )
        )

        if not candidate_sources:
            continue

        current_max_sources = (
            len(candidate_sources)
            if max_sources is None
            else min(
                max_sources,
                len(candidate_sources),
            )
        )

        if min_sources > current_max_sources:
            continue

        target_domain_ids = get_domain_ids(
            dataframe=selected_df,
            dataset=target_dataset,
            subject=target_subject,
        )

        for repetition in range(n_repetitions):
            split_seed = (
                seed
                + target_index * n_repetitions
                + repetition
            )

            rng = np.random.default_rng(
                split_seed
            )

            source_order = [
                candidate_sources[index]
                for index in rng.permutation(
                    len(candidate_sources)
                )
            ]

            for n_sources in range(
                min_sources,
                current_max_sources + 1,
            ):
                selected_sources = source_order[
                    :n_sources
                ]

                source_domain_ids = []

                for (
                    source_dataset,
                    source_subject,
                ) in selected_sources:
                    source_domain_ids.extend(
                        get_domain_ids(
                            dataframe=selected_df,
                            dataset=source_dataset,
                            subject=source_subject,
                        )
                    )

                split_id = (
                    "increasing_source_domains__"
                    f"target_"
                    f"{target_dataset}_{target_subject}"
                    f"__n_{n_sources}"
                    f"__rep_{repetition}"
                )

                splits.append(
                    create_split_record(
                        strategy=(
                            "increasing_source_domains"
                        ),
                        split_id=split_id,
                        source_domain_ids=(
                            source_domain_ids
                        ),
                        target_domain_ids=(
                            target_domain_ids
                        ),
                        seed=split_seed,
                        repetition=repetition,
                        metadata={
                            "target_dataset": (
                                target_dataset
                            ),
                            "target_subject": (
                                target_subject
                            ),
                            "n_source_subjects": (
                                n_sources
                            ),
                            "same_dataset_only": (
                                same_dataset_only
                            ),
                        },
                    )
                )

    return splits

In [20]:
# ============================================================
# Session-level strategies
# ============================================================

def generate_cross_session_pooled_splits(
    dataframe,
    datasets=None,
):
    """
    Pool subjects within each dataset and transfer from one
    session to another.

    Example:

        all session_01 domains -> all session_02 domains
    """
    selected_df = select_datasets(
        dataframe=dataframe,
        datasets=datasets,
    )

    splits = []

    for dataset in sorted(
        selected_df["dataset"].unique()
    ):
        dataset_df = selected_df.loc[
            selected_df["dataset"] == dataset
        ]

        sessions = sorted(
            dataset_df["session"]
            .unique()
            .tolist()
        )

        if len(sessions) < 2:
            print(
                f"Skipping {dataset}: fewer than "
                "two sessions."
            )
            continue

        for (
            source_session,
            target_session,
        ) in permutations(
            sessions,
            2,
        ):
            source_domain_ids = get_domain_ids(
                dataframe=dataset_df,
                dataset=dataset,
                session=source_session,
            )

            target_domain_ids = get_domain_ids(
                dataframe=dataset_df,
                dataset=dataset,
                session=target_session,
            )

            split_id = (
                "cross_session_pooled__"
                f"{dataset}__"
                f"{source_session}_to_{target_session}"
            )

            splits.append(
                create_split_record(
                    strategy="cross_session_pooled",
                    split_id=split_id,
                    source_domain_ids=(
                        source_domain_ids
                    ),
                    target_domain_ids=(
                        target_domain_ids
                    ),
                    metadata={
                        "dataset": dataset,
                        "source_session": (
                            source_session
                        ),
                        "target_session": (
                            target_session
                        ),
                    },
                )
            )

    return splits


def generate_cross_session_within_subject_splits(
    dataframe,
    datasets=None,
):
    """
    Transfer between sessions belonging to the same subject.

    Example:

        A01 session_01 -> A01 session_02
        A01 session_02 -> A01 session_01
    """
    selected_df = select_datasets(
        dataframe=dataframe,
        datasets=datasets,
    )

    subject_units = _get_subject_units(
        selected_df
    )

    splits = []

    for (
        dataset,
        subject,
    ) in subject_units.itertuples(
        index=False,
        name=None,
    ):
        subject_df = selected_df.loc[
            (
                selected_df["dataset"]
                == dataset
            )
            & (
                selected_df["subject"]
                == subject
            )
        ]

        sessions = sorted(
            subject_df["session"]
            .unique()
            .tolist()
        )

        if len(sessions) < 2:
            continue

        for (
            source_session,
            target_session,
        ) in permutations(
            sessions,
            2,
        ):
            source_domain_ids = get_domain_ids(
                dataframe=subject_df,
                dataset=dataset,
                subject=subject,
                session=source_session,
            )

            target_domain_ids = get_domain_ids(
                dataframe=subject_df,
                dataset=dataset,
                subject=subject,
                session=target_session,
            )

            split_id = (
                "cross_session_within_subject__"
                f"{dataset}__{subject}__"
                f"{source_session}_to_{target_session}"
            )

            splits.append(
                create_split_record(
                    strategy=(
                        "cross_session_within_subject"
                    ),
                    split_id=split_id,
                    source_domain_ids=(
                        source_domain_ids
                    ),
                    target_domain_ids=(
                        target_domain_ids
                    ),
                    metadata={
                        "dataset": dataset,
                        "subject": subject,
                        "source_session": (
                            source_session
                        ),
                        "target_session": (
                            target_session
                        ),
                    },
                )
            )

    return splits

In [21]:
# ============================================================
# Strategy configuration
# ============================================================

STRATEGY = "cross_dataset_leave_one_out"

# None uses every available dataset.
SELECTED_DATASETS = None


# ============================================================
# Generate split specifications
# ============================================================

if STRATEGY == "within_subject_pooled":

    domain_splits = (
        generate_within_subject_pooled_splits(
            dataframe=domain_df,
            datasets=SELECTED_DATASETS,
        )
    )


elif STRATEGY == "within_session":

    domain_splits = generate_within_session_splits(
        dataframe=domain_df,
        datasets=SELECTED_DATASETS,
    )


elif STRATEGY == "cross_dataset_leave_one_out":

    domain_splits = (
        generate_cross_dataset_leave_one_out_splits(
            dataframe=domain_df,
            datasets=SELECTED_DATASETS,
        )
    )


elif STRATEGY == "cross_dataset_pairwise":

    domain_splits = (
        generate_cross_dataset_pairwise_splits(
            dataframe=domain_df,
            datasets=SELECTED_DATASETS,
        )
    )


elif STRATEGY == "cross_session_pooled":

    domain_splits = (
        generate_cross_session_pooled_splits(
            dataframe=domain_df,
            datasets=SELECTED_DATASETS,
        )
    )


elif STRATEGY == "cross_session_within_subject":

    domain_splits = (
        generate_cross_session_within_subject_splits(
            dataframe=domain_df,
            datasets=SELECTED_DATASETS,
        )
    )


elif STRATEGY == "leave_one_subject_out":

    domain_splits = (
        generate_leave_one_subject_out_splits(
            dataframe=domain_df,
            datasets=SELECTED_DATASETS,

            # True restricts source subjects to the
            # target subject's dataset.
            same_dataset_only=False,
        )
    )


elif STRATEGY == "pairwise_subjects":

    domain_splits = (
        generate_pairwise_subject_splits(
            dataframe=domain_df,
            datasets=SELECTED_DATASETS,
            same_dataset_only=False,

            # None generates every ordered pair.
            max_pairs=None,
            seed=42,
        )
    )


elif STRATEGY == "increasing_source_domains":

    domain_splits = (
        generate_increasing_source_domain_splits(
            dataframe=domain_df,
            datasets=SELECTED_DATASETS,
            same_dataset_only=False,
            n_repetitions=1,
            min_sources=1,
            max_sources=None,
            seed=42,
        )
    )


else:
    raise ValueError(
        f"Unknown strategy: {STRATEGY}"
    )


# ============================================================
# Validate and summarize generated splits
# ============================================================

validate_domain_splits(
    splits=domain_splits,
    dataframe=domain_df,
)

split_summary = create_split_summary(
    domain_splits
)

print(
    f"✅ Strategy: {STRATEGY}"
)

print(
    f"✅ Generated splits: "
    f"{len(domain_splits)}"
)

display(split_summary.head(20))


# ============================================================
# Select one split for inspection
# ============================================================

SPLIT_INDEX = 0

if not 0 <= SPLIT_INDEX < len(domain_splits):
    raise IndexError(
        f"SPLIT_INDEX must be between 0 and "
        f"{len(domain_splits) - 1}."
    )

selected_split = domain_splits[
    SPLIT_INDEX
]

print("\nSelected split")
display(
    pd.Series(selected_split)
)


# ============================================================
# Materialize selected split
# ============================================================

materialized = materialize_domain_split(
    dataframe=domain_df,
    split=selected_split,
)

source_df = materialized["source_df"]
target_df = materialized["target_df"]
within_df = materialized["within_df"]


if selected_split["requires_trial_split"]:

    print(
        "\nThis is a within-domain strategy."
    )

    print(
        "The trials will be divided into source and target "
        "later using a training/evaluation split."
    )

    print(
        f"Available trials: {len(within_df)}"
    )

    display(
        within_df[
            [
                "dataset",
                "domain_id",
                "subject",
                "session",
                "label",
            ]
        ]
        .head()
    )


else:

    print(
        f"\nSource trials: {len(source_df)}"
    )

    print(
        f"Target trials: {len(target_df)}"
    )

    print(
        f"Source elementary domains: "
        f"{source_df['domain_id'].nunique()}"
    )

    print(
        f"Target elementary domains: "
        f"{target_df['domain_id'].nunique()}"
    )

    print("\nSource labels")
    display(
        source_df["label"]
        .value_counts()
        .sort_index()
        .rename("n_trials")
    )

    print("Target labels")
    display(
        target_df["label"]
        .value_counts()
        .sort_index()
        .rename("n_trials")
    )

✅ Strategy: cross_dataset_leave_one_out
✅ Generated splits: 2


,split_id,strategy,n_source_domains,n_target_domains,n_shared_domains,requires_trial_split,seed,repetition,source_datasets,target_dataset
0,cross_dataset_leave_one_out__target_bci_iv_2a,cross_dataset_leave_one_out,109,18,0,False,None,None,eegmmidb,bci_iv_2a
1,cross_dataset_leave_one_out__target_eegmmidb,cross_dataset_leave_one_out,18,109,0,False,None,None,bci_iv_2a,eegmmidb



Selected split


split_id                    cross_dataset_leave_one_out__target_bci_iv_2a
strategy                                      cross_dataset_leave_one_out
source_domain_ids       [eegmmidb_S001_session_01, eegmmidb_S002_sessi...
target_domain_ids       [bci_iv_2a_A01_session_01, bci_iv_2a_A01_sessi...
shared_domain_ids                                                      []
requires_trial_split                                                False
n_source_domains                                                      109
n_target_domains                                                       18
n_shared_domains                                                        0
seed                                                                 None
repetition                                                           None
metadata                {'source_datasets': 'eegmmidb', 'target_datase...
dtype: object


Source trials: 19673
Target trials: 5184
Source elementary domains: 109
Target elementary domains: 18

Source labels


label
both_feet_execution     2469
both_feet_imagery       2455
both_fists_execution    2440
both_fists_imagery      2465
left_fist_execution     2471
left_fist_imagery       2479
right_fist_execution    2456
right_fist_imagery      2438
Name: n_trials, dtype: int64

Target labels


label
both_feet_imagery     1296
left_hand_imagery     1296
right_hand_imagery    1296
tongue_imagery        1296
Name: n_trials, dtype: int64